# Cross-validation for model selection (LogReg vs RF vs XGBoost) and single held-out test evaluation

Workflow:
1. Recreate the same train/val/test split by pdb_id used elsewhere in the project (random_state=42), so thetest set is identical to the rest of the project and excluded from CV entirely.
2. Run `GroupKFold` (grouped by `pdb_id`) on the train+val comparing Logistic Regression, Random Forest, and XGBoost, each as a full sklearn pipelin with its own preprocessing, matching notebooks 3 and 7.
3. Pick the model with the best mean CV score
4. Refit the winning pipeline on the full train+val .
5. Evaluate once on the untouched test set,this is the only number that should be reported as final test performance.

In [1]:
import os
import gc
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    matthews_corrcoef,
    balanced_accuracy_score,
    f1_score,
    average_precision_score,
    roc_auc_score
)

# Drive path for Google Colab
#from google.colab import drive
#drive.mount('/content/drive')
#df = pd.read_parquet('/content/drive/MyDrive/SB/SB_Project/classification_ring/data/processed/data_ml.parquet')

# Local path
df = pd.read_parquet('classification_ring/data/processed/data_ml.parquet')

num_features = [
    's_rsa', 's_phi', 's_psi', 's_a1', 's_a2', 's_a3', 's_a4', 's_a5',
    't_rsa', 't_phi', 't_psi', 't_a1', 't_a2', 't_a3', 't_a4', 't_a5'
]
cat_features = ['s_ss8', 's_3di_letter', 't_ss8', 't_3di_letter']
feature_cols = num_features + cat_features
label_cols = ['HBOND', 'VDW', 'IONIC', 'PIPISTACK', 'PICATION', 'SSBOND', 'PIHBOND']

df[num_features] = df[num_features].astype('float32')
print(df.shape)

(1452411, 38)


## Step 1: Recreate the same train/val/test split (same seed as notebooks 2 / 3 / 7)

Guarantees the test set is identical to the one already used elsewhere, and excluded from everything below.

In [3]:
pdb_ids = df['pdb_id'].unique().to_numpy()
print(f"Total unique PDBs: {len(pdb_ids)}")

train_pdbs, temp_pdbs = train_test_split(pdb_ids, test_size=0.30, random_state=42)
val_pdbs, test_pdbs   = train_test_split(temp_pdbs, test_size=0.50, random_state=42)

trainval_pdbs = np.concatenate([train_pdbs, val_pdbs])

trainval_df = df[df['pdb_id'].isin(trainval_pdbs)].reset_index(drop=True)
test_df     = df[df['pdb_id'].isin(test_pdbs)].reset_index(drop=True)

print(f"Train+Val pool: {len(trainval_df)} rows, {len(trainval_pdbs)} PDBs  <- used for CV / model selection")
print(f"Test (held out, untouched until the end): {len(test_df)} rows, {len(test_pdbs)} PDBs")

del df
gc.collect()

Total unique PDBs: 3827
Train+Val pool: 1237893 rows, 3252 PDBs  <- used for CV / model selection
Test (held out, untouched until the end): 214518 rows, 575 PDBs


1221

In [4]:
def make_preprocessor():
    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    return ColumnTransformer([
        ('num', num_pipeline, num_features),
        ('cat', cat_pipeline, cat_features)
    ])


def make_logreg_pipeline():
    return Pipeline([
        ('preprocessor', make_preprocessor()),
        ('classifier', MultiOutputClassifier(
            LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
        ))
    ])


def make_rf_pipeline():
    return Pipeline([
        ('preprocessor', make_preprocessor()),
        ('classifier', MultiOutputClassifier(
            RandomForestClassifier(
                n_estimators=100, max_depth=20,
                class_weight='balanced', random_state=42, n_jobs=2
            )
        ))
    ])


def make_xgb_pipeline():
    return Pipeline([
        ('preprocessor', make_preprocessor()),
        ('classifier', MultiOutputClassifier(
            xgb.XGBClassifier(
                n_estimators=150, max_depth=6, learning_rate=0.1,
                tree_method='hist', random_state=42, n_jobs=2
            )
        ))
    ])


candidate_builders = {
    'logreg': make_logreg_pipeline,
    'rf': make_rf_pipeline,
    'xgboost': make_xgb_pipeline,
}

In [5]:
THRESHOLDS = np.arange(0.00, 1.01, 0.01)

def evaluate_predictions(Y_true_df, Y_proba_list, label_cols, thresholds=None):
    """
    If thresholds=None:
        Optimize the threshold for each class by maximizing MCC.
    Otherwise:
        Use the supplied threshold for each class.
    """
    out = {}

    if thresholds is None:
        thresholds = {}

    for i, label in enumerate(label_cols):

        y_true = Y_true_df[label].values
        proba = Y_proba_list[i]
        y_score = proba[:, 1] if proba.shape[1] > 1 else np.zeros_like(y_true)


        if label not in thresholds:

            best_threshold = 0.5
            best_mcc = -1

            for t in THRESHOLDS:
                y_pred = (y_score >= t).astype(int)
                mcc = matthews_corrcoef(y_true, y_pred)

                if mcc > best_mcc:
                    best_mcc = mcc
                    best_threshold = t

            thresholds[label] = best_threshold


        t = thresholds[label]
        y_pred = (y_score >= t).astype(int)

        out[f'{label}_threshold'] = t
        out[f'{label}_mcc'] = matthews_corrcoef(y_true, y_pred)
        out[f'{label}_bal_acc'] = balanced_accuracy_score(y_true, y_pred)
        out[f'{label}_f1'] = f1_score(y_true, y_pred, zero_division=0)

        try:
            out[f'{label}_ap'] = average_precision_score(y_true, y_score)
            out[f'{label}_auroc'] = roc_auc_score(y_true, y_score)
        except ValueError:
            out[f'{label}_ap'] = np.nan
            out[f'{label}_auroc'] = np.nan

    return out, thresholds

## Step 3: run GroupKFold CV for each candidate model (on train+val)

In [7]:
N_SPLITS = 5
gkf = GroupKFold(n_splits=N_SPLITS)

X_full = trainval_df[feature_cols]
Y_full = trainval_df[label_cols]
groups = trainval_df['pdb_id'].values

all_fold_rows = []

for model_name, builder in candidate_builders.items():
    print(f"\n=== Model: {model_name} ===")
    for fold_idx, (train_idx, val_idx) in enumerate(gkf.split(X_full, Y_full, groups=groups)):
        X_train = X_full.iloc[train_idx]
        X_val   = X_full.iloc[val_idx]
        Y_train = Y_full.iloc[train_idx].reset_index(drop=True)
        Y_val   = Y_full.iloc[val_idx].reset_index(drop=True)

        pipeline = builder()
        pipeline.fit(X_train, Y_train)

        Y_proba = pipeline.predict_proba(X_val)
        # compute validation metrics at optimal thresholds
        metrics, fold_thresholds = evaluate_predictions(
            Y_val,
            Y_proba,
            label_cols
            )
        metrics["model"] = model_name
        metrics["fold"] = fold_idx
        all_fold_rows.append(metrics)

        print(f"  fold {fold_idx} done")

        del pipeline, X_train, X_val, Y_train, Y_val,  Y_proba
        gc.collect()

cv_results = pd.DataFrame(all_fold_rows)
cv_results


=== Model: logreg ===
  fold 0 done
  fold 1 done
  fold 2 done
  fold 3 done
  fold 4 done

=== Model: rf ===
  fold 0 done
  fold 1 done
  fold 2 done
  fold 3 done
  fold 4 done

=== Model: xgboost ===
  fold 0 done
  fold 1 done
  fold 2 done
  fold 3 done
  fold 4 done


,HBOND_threshold,HBOND_mcc,HBOND_bal_acc,HBOND_f1,HBOND_ap,HBOND_auroc,VDW_threshold,VDW_mcc,VDW_bal_acc,VDW_f1,...,SSBOND_ap,SSBOND_auroc,PIHBOND_threshold,PIHBOND_mcc,PIHBOND_bal_acc,PIHBOND_f1,PIHBOND_ap,PIHBOND_auroc,model,fold
0,0.42,0.230733,0.611623,0.798313,0.827736,0.669155,0.49,0.130208,0.564949,0.583410,...,0.826569,0.999838,0.83,0.079931,0.776449,0.023674,0.011736,0.927934,logreg,0
1,0.40,0.235380,0.609239,0.809461,0.829494,0.672485,0.49,0.131952,0.565655,0.590345,...,0.895240,0.999830,0.83,0.076991,0.763417,0.023199,0.013712,0.913600,logreg,1
2,0.39,0.222010,0.598490,0.818979,0.830894,0.667093,0.53,0.128352,0.560801,0.477993,...,0.706790,0.999723,0.76,0.064871,0.754357,0.017346,0.009710,0.897881,logreg,2
3,0.40,0.234476,0.609069,0.811151,0.832729,0.673954,0.51,0.131209,0.565288,0.545076,...,0.875446,0.999857,0.81,0.064905,0.756216,0.017153,0.010783,0.901267,logreg,3
4,0.42,0.234881,0.613428,0.801156,0.829191,0.671345,0.49,0.125887,0.562715,0.584953,...,0.852564,0.999890,0.55,0.065528,0.820915,0.013915,0.010260,0.902025,logreg,4
5,0.48,0.359886,0.672740,0.833722,0.878360,0.756105,0.49,0.202600,0.600542,0.627266,...,0.848170,0.999866,0.37,0.098548,0.798795,0.032592,0.035710,0.945962,rf,0
6,0.49,0.364764,0.680211,0.828705,0.879088,0.758260,0.51,0.206623,0.602888,0.586514,...,0.921016,0.999867,0.39,0.102533,0.775274,0.038058,0.027256,0.946032,rf,1
7,0.51,0.365367,0.687798,0.821722,0.884172,0.759852,0.52,0.207865,0.600806,0.546332,...,0.828957,0.999833,0.89,0.140773,0.513646,0.052632,0.053240,0.939564,rf,2
8,0.46,0.371124,0.671022,0.845852,0.882955,0.761069,0.51,0.203838,0.601450,0.583373,...,0.888051,0.999888,0.90,0.105943,0.505618,0.022222,0.032916,0.947728,rf,3
9,0.48,0.371755,0.678572,0.837687,0.882322,0.761577,0.50,0.201556,0.600790,0.605471,...,0.902786,0.999917,0.20,0.100545,0.875790,0.026951,0.021882,0.947516,rf,4


## Step 4: pick the best model by mean macro-MCC across folds

In [8]:
mcc_cols = [c for c in cv_results.columns if c.endswith('_mcc')]
cv_results['macro_mcc'] = cv_results[mcc_cols].mean(axis=1)

model_summary = cv_results.groupby('model')['macro_mcc'].agg(['mean', 'std']).sort_values('mean', ascending=False)
print(model_summary)

best_model_name = model_summary.index[0]
best_cv = cv_results[cv_results["model"] == best_model_name]

final_thresholds = {
    label: best_cv[f"{label}_threshold"].mean()
    for label in label_cols
}

print("Final thresholds:")
print(final_thresholds)
print(f"\nBest model: {best_model_name}")

             mean       std
model                      
xgboost  0.494863  0.003593
rf       0.491438  0.002546
logreg   0.377164  0.006749
Final thresholds:
{'HBOND': np.float64(0.634), 'VDW': np.float64(0.526), 'IONIC': np.float64(0.192), 'PIPISTACK': np.float64(0.43800000000000006), 'PICATION': np.float64(0.122), 'SSBOND': np.float64(0.40800000000000003), 'PIHBOND': np.float64(0.05600000000000001)}

Best model: xgboost


## Step 5: refit the winning model's pipeline on the full train+val pool

In [9]:
final_pipeline = candidate_builders[best_model_name]()
final_pipeline.fit(X_full, Y_full)
print(f"Final '{best_model_name}' pipeline fit on full train+val pool.")

Final 'xgboost' pipeline fit on full train+val pool.


In [10]:
X_test = test_df[feature_cols]
Y_test = test_df[label_cols].reset_index(drop=True)

Y_test_proba = final_pipeline.predict_proba(X_test)

test_metrics, _ = evaluate_predictions(
    Y_test,
    Y_test_proba,
    label_cols,
    thresholds=final_thresholds
)
test_metrics_df = pd.DataFrame([test_metrics])
test_metrics_df

,HBOND_threshold,HBOND_mcc,HBOND_bal_acc,HBOND_f1,HBOND_ap,HBOND_auroc,VDW_threshold,VDW_mcc,VDW_bal_acc,VDW_f1,...,SSBOND_bal_acc,SSBOND_f1,SSBOND_ap,SSBOND_auroc,PIHBOND_threshold,PIHBOND_mcc,PIHBOND_bal_acc,PIHBOND_f1,PIHBOND_ap,PIHBOND_auroc
0,0.634,0.356632,0.663203,0.840428,0.876795,0.753483,0.526,0.19915,0.597676,0.55694,...,0.981242,0.873596,0.924638,0.999902,0.056,0.085486,0.557579,0.083218,0.057584,0.954577


In [11]:

#out_dir = '/content/drive/MyDrive/SB/SB_Project/classification_ring/results'
out_dir = 'classification_ring/results'
os.makedirs(out_dir, exist_ok=True)

cv_results.to_csv(os.path.join(out_dir, 'model_selection_cv_per_fold.csv'), index=False)
model_summary.to_csv(os.path.join(out_dir, 'model_selection_cv_summary.csv'))
test_metrics_df.to_csv(os.path.join(out_dir, 'final_test_metrics.csv'), index=False)

print("Saved: per-fold CV results, model comparison summary, and final test metrics.")

Saved: per-fold CV results, model comparison summary, and final test metrics.
